In [94]:
import pandas as pd

# 1. Load your downloaded CSV file
# IMPORTANT: Change 'your_downloaded_file.csv' to the actual name of your file!
file_name = "../data/gdelt/April16th_w_ELECTION_PROPAGANDA.csv"
print(f"Loading data from {file_name}...\n")
df = pd.read_csv(file_name)
df.head()


# CLEANING THE DATA: remove duplicate rows
rows_before = len(df)
print(f"Number of rows before removing duplicates: {rows_before}")

df.drop_duplicates(inplace=True)

rows_after = len(df)
print(f"Number of rows after removing duplicates: {rows_after}")


Loading data from ../data/gdelt/April16th_w_ELECTION_PROPAGANDA.csv...

Number of rows before removing duplicates: 2943
Number of rows after removing duplicates: 2281


In [95]:
# Save counts and marks for each country, order by day
df['Date'] = pd.to_datetime(df['Event_Date'].astype(str), format='%Y%m%d', errors='coerce')
daily_counts = df.groupby(['Date', 'Initiator_Country'])['Date'].value_counts().reset_index(name='News')
daily_counts = daily_counts.sort_values('Date')

daily_marks = df.groupby(['Date', 'Initiator_Country'])['Total_Mentions'].sum().reset_index()
daily_marks = daily_marks.sort_values('Date')

# Concat
daily_data = pd.merge(daily_counts, daily_marks, on=['Date', 'Initiator_Country'], how='left')

daily_data.to_csv('daily_news_data.csv', index=False)

daily_data.head()

,Date,Initiator_Country,News,Total_Mentions
0,2024-01-01,CHN,1,1
1,2024-01-02,CHN,1,1
2,2024-01-02,RUS,3,13
3,2024-01-04,CHN,1,5
4,2024-01-05,CHN,2,10


In [96]:
daily_counts.sort_values('News', ascending=False).head(10)

,Date,Initiator_Country,News
386,2024-09-05,RUS,57
388,2024-09-06,RUS,41
47,2024-02-09,RUS,22
108,2024-03-14,CHN,22
69,2024-02-22,RUS,22
186,2024-04-24,CHN,20
429,2024-09-26,RUS,20
477,2024-10-25,RUS,18
501,2024-11-06,RUS,16
45,2024-02-08,RUS,16


#### Temporal plots

In [97]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

daily_df = pd.read_csv('daily_news_data.csv')

# Ensure the Date column is recognized as actual datetime objects
daily_df['Date'] = pd.to_datetime(daily_df['Date'])

def plot_line_charts(daily_df, output_name = 'charts'):

    # Set up our consistent colors
    colors = {'RUS': '#d62728', 'CHN': '#1f77b4', 'IRN': '#2ca02c'}

    # =========================================================
    # PLOT 1: ALL COUNTRIES ON THE SAME AXIS
    # =========================================================
    plt.figure(figsize=(15, 6))
    sns.lineplot(
        data=daily_df, 
        x='Date', 
        y='News', 
        hue='Initiator_Country',
        palette=colors,
        linewidth=2.5,  # Make lines thicker
        marker='o',     # Add a dot for each week
        markersize=6
    )

    plt.title('Daily Election Interference News (Combined Axis)', fontsize=18, fontweight='bold', pad=20)
    plt.xlabel('Date (2024)', fontsize=14)
    plt.ylabel('Total News News (Daily)', fontsize=14)

    # Force ticks to appear on months, formatted as abbreviated month names (Jan, Feb)
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b'))

    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(title='Actor', loc='upper right')

    plt.tight_layout()
    output_1 = f'model_plots/{output_name}_combined_2024.png'
    plt.savefig(output_1, dpi=300)
    print(f"Combined line chart saved as: {output_1}")
    plt.close() # Close to start the next plot fresh


    # =========================================================
    # PLOT 2: SEPARATED "LANES" (SUBPLOTS)
    # =========================================================
    # Create 3 rows and 1 column of charts, sharing the same X-axis
    fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

    country_names = {'RUS': 'Russia', 'CHN': 'China', 'IRN': 'Iran'}

    # Loop through our three countries and plot them on their own axis
    for i, country in enumerate(['RUS', 'CHN', 'IRN']):
        # Filter the data for just this country
        country_data = daily_df[daily_df['Initiator_Country'] == country]
        
        sns.lineplot(
            data=country_data,
            x='Date',
            y='News',
            color=colors[country],
            ax=axes[i],
            linewidth=2.5,
            marker='o',
            markersize=6
        )
        
        # Format each individual subplot
        axes[i].set_title(f'{country_names[country]} Activity', fontsize=14, fontweight='bold', color=colors[country])
        axes[i].set_ylabel('Daily News', fontsize=12)
        axes[i].set_ylim(0, daily_df['News'].max() * 1.05)  # Set a consistent Y-axis limit for better comparison
        axes[i].grid(True, linestyle='--', alpha=0.5)
        
        # --- NEW: Apply month formatting to EVERY axis ---
        axes[i].xaxis.set_major_locator(mdates.MonthLocator())
        axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%b'))
        
        # --- NEW: Force the month labels to appear despite sharex=True ---
        axes[i].tick_params(axis='x', labelbottom=True)
        
        # Remove the generic "Date" x-axis label for the top two plots, but keep the months
        if i < 2:
            axes[i].set_xlabel('')
        else:
            axes[i].set_xlabel('Timeline (2024)', fontsize=14)

    #plt.suptitle('Daily Election Interference News (Separated by Actor)', fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()

    output_2 = f'model_plots/{output_name}_separated_2024.png'
    plt.savefig(output_2, dpi=300)
    print(f"Separated line chart saved as: {output_2}")
    plt.close()

In [98]:
plot_line_charts(daily_df, output_name='all_news')


Combined line chart saved as: model_plots/all_news_combined_2024.png
Separated line chart saved as: model_plots/all_news_separated_2024.png


In [99]:
# Filter the column Themes to check whether the theme "ELECTION" is present
election_df = df[df['Themes'].str.contains('ELECTION', na=False)]

election_df['Date'] = pd.to_datetime(election_df['Event_Date'].astype(str), format='%Y%m%d', errors='coerce')
daily_counts = election_df.groupby(['Date', 'Initiator_Country'])['Date'].value_counts().reset_index(name='News')
daily_counts = daily_counts.sort_values('Date')

daily_marks = election_df.groupby(['Date', 'Initiator_Country'])['Total_Mentions'].sum().reset_index()
daily_marks = daily_marks.sort_values('Date')

# Concat
daily_data = pd.merge(daily_counts, daily_marks, on=['Date', 'Initiator_Country'], how='left')

plot_line_charts(daily_data, output_name='election_news')

Combined line chart saved as: model_plots/election_news_combined_2024.png
Separated line chart saved as: model_plots/election_news_separated_2024.png


#### Check on mentioned themes

In [100]:
import re
# 2. Define the EXACT themes from your SQL WHERE clause
themes_of_interest = [


    '''r'INTERFERE',
    
    # Cyber Operations
     r'HACK', r'ESPIONAGE', r'LEAK', r'CYBERATTACK', r'CYBEROPERATION', r'CYBER_ATTACK', r"CYBER_SECURITY",
    
    # Disinformation
    r'DISINFORMATION', r'MISINFORMATION', r'FAKENEWS', 
    r'DEEPFAKE', r'CONSPIRACY', r'HOAX', r'PROPAGANDA',
    
    # Bots & Influence
    r'INFLUENCE_OP', r'TROLL_FARM', r'BOTNET', r'ASTROTURF', r'SPAMOUFLAGE',

    r'ELECTION'''

    r'CYBER_ATTACK', r'CYBERATTACK', r'PROPAGANDA', r'ELECTION', r'MEDIA_SOCIAL', r'PIRACY', r'PROPAGANDA', r'SURVEILLANCE', r'WHISTLEBLOWER'
    r'INFORMATION', r'INFO_HOAX',

    r'HACKERS', r'ICT_SECURITY'
]

# 3. Compile the search pattern (ignoring case just to be safe)
pattern = re.compile(r'(' + '|'.join(themes_of_interest) + r')', flags=re.IGNORECASE)

# 4. Function to extract and clean the themes
def extract_specific_themes(text):
    if pd.isna(text):
        return None
        
    # Find all matches in the text
    matches = pattern.findall(str(text))
    
    if not matches:
        return None
        
    # Clean up: uppercase, remove duplicates, and sort them alphabetically
    unique_matches = sorted(list(set([match.upper() for match in matches])))
    
    # Return as a clean, comma-separated string
    return ", ".join(unique_matches)

# 5. Apply the function to create your new column
df['Detected_Themes'] =   df['Themes'].apply(extract_specific_themes)

In [101]:
# Count the occurrences of each theme in the Detected_Themes column
theme_counts = df['Detected_Themes'].value_counts(dropna=True)
theme_counts.head(20)

Detected_Themes
ELECTION, PROPAGANDA                                              1074
ELECTION, MEDIA_SOCIAL, PROPAGANDA                                 701
ELECTION, ICT_SECURITY, MEDIA_SOCIAL, PROPAGANDA                   109
ELECTION, MEDIA_SOCIAL, PROPAGANDA, SURVEILLANCE                    58
ELECTION, PROPAGANDA, SURVEILLANCE                                  48
ELECTION, HACKERS, ICT_SECURITY, MEDIA_SOCIAL, PROPAGANDA           41
ELECTION, INFO_HOAX, MEDIA_SOCIAL, PROPAGANDA                       35
ELECTION, ICT_SECURITY, PROPAGANDA                                  25
ELECTION, PIRACY, PROPAGANDA                                        25
ELECTION, MEDIA_SOCIAL, PIRACY, PROPAGANDA                          24
ELECTION, HACKERS, PROPAGANDA                                       20
ELECTION, HACKERS, MEDIA_SOCIAL, PROPAGANDA                         17
ELECTION, INFO_HOAX, PROPAGANDA                                     15
ELECTION, ICT_SECURITY, MEDIA_SOCIAL, PIRACY, PROPAGANDA     

In [102]:
# Display the results
df[['News_URL', 'Detected_Themes']]

,News_URL,Detected_Themes
0,https://www.yahoo.com/news/forget-cats-eat-hum...,"ELECTION, MEDIA_SOCIAL, PROPAGANDA"
2,https://orainfo.net/agjencite-federale-thone-s...,"ELECTION, PROPAGANDA"
3,https://news.yahoo.com/ex-fbi-informant-charge...,"ELECTION, PROPAGANDA"
4,https://www.hometownregister.com/news/national...,"ELECTION, MEDIA_SOCIAL, PROPAGANDA"
5,https://www.kcci.com/article/russia-iran-rampi...,"ELECTION, HACKERS, ICT_SECURITY, MEDIA_SOCIAL,..."
...,...,...
2937,https://english.aawsat.com/world/5057827-putin...,"ELECTION, PROPAGANDA"
2938,https://timesofindia.indiatimes.com/world/us/g...,"ELECTION, PROPAGANDA"
2940,https://www.euractiv.com/section/politics/news...,"ELECTION, MEDIA_SOCIAL, PROPAGANDA"
2941,https://www.clevelandjewishnews.com/jns/nj-geo...,"ELECTION, PROPAGANDA"


In [103]:
# Get the rows with detected_themes that contain "FAKE_NEWS"
fake_news_df = df[df['Detected_Themes'].str.contains('CYBER', na=False)]
fake_news_df[['News_URL', 'Detected_Themes']]

,News_URL,Detected_Themes


In [104]:
df[['News_URL', 'Detected_Themes']].value_counts(subset=['Detected_Themes']).head(20)

Detected_Themes                                               
ELECTION, PROPAGANDA                                              1074
ELECTION, MEDIA_SOCIAL, PROPAGANDA                                 701
ELECTION, ICT_SECURITY, MEDIA_SOCIAL, PROPAGANDA                   109
ELECTION, MEDIA_SOCIAL, PROPAGANDA, SURVEILLANCE                    58
ELECTION, PROPAGANDA, SURVEILLANCE                                  48
ELECTION, HACKERS, ICT_SECURITY, MEDIA_SOCIAL, PROPAGANDA           41
ELECTION, INFO_HOAX, MEDIA_SOCIAL, PROPAGANDA                       35
ELECTION, ICT_SECURITY, PROPAGANDA                                  25
ELECTION, PIRACY, PROPAGANDA                                        25
ELECTION, MEDIA_SOCIAL, PIRACY, PROPAGANDA                          24
ELECTION, HACKERS, PROPAGANDA                                       20
ELECTION, HACKERS, MEDIA_SOCIAL, PROPAGANDA                         17
ELECTION, INFO_HOAX, PROPAGANDA                                     15
ELECTION, ICT_

#### Webscrapping

In [105]:
df.head()

,Event_Date,Initiator,Initiator_Country,Target,Total_Mentions,Unique_Outlets,Total_Articles,Social_Media_Mentioned,All_Organizations_Found,Themes,News_URL,Date,Detected_Themes
0,20240913,CHINA,CHN,UNITED STATES,44,1,44,NaN,"Sandy Hook Elementary School,4051;Justice Depa...","TAX_WORLDMAMMALS_HUMANS,1053;EPU_POLICY_WHITE_...",https://www.yahoo.com/news/forget-cats-eat-hum...,2024-09-13,"ELECTION, MEDIA_SOCIAL, PROPAGANDA"
2,20241105,RUSSIA,RUS,PENNSYLVANIA,40,4,40,NaN,"Embassy Russian,1231;Microsoft,2668;Agency Sec...","TAX_FNCACT_ANALYSTS,2643;TAX_DISEASE_OBESE,800...",https://orainfo.net/agjencite-federale-thone-s...,2024-11-05,"ELECTION, PROPAGANDA"
3,20240221,RUSSIAN,RUS,MARYLAND,30,6,30,NaN,"Associated Press,6259;Judiciary Committee,5986...","ECON_STOCKMARKET,1334;ARMEDCONFLICT,4853;TAX_F...",https://news.yahoo.com/ex-fbi-informant-charge...,2024-02-21,"ELECTION, PROPAGANDA"
4,20240420,BEIJING,CHN,THE US,30,3,30,NaN,"Senate Democratic,2250;United States,896;Unite...","DEMOCRACY,1133;DEMOCRACY,2250;CONFISCATION,203...",https://www.hometownregister.com/news/national...,2024-04-20,"ELECTION, MEDIA_SOCIAL, PROPAGANDA"
5,20241006,RUSSIA,RUS,WASHINGTON,24,3,24,NaN,"Infrastructure Security Agency,1188;National I...","TAX_ETHNICITY_IRANIAN,2609;TAX_ETHNICITY_IRANI...",https://www.kcci.com/article/russia-iran-rampi...,2024-10-06,"ELECTION, HACKERS, ICT_SECURITY, MEDIA_SOCIAL,..."


In [106]:
import pandas as pd
from newspaper import Article
import spacy

# 1. Load the NLP model from spaCy (to understand English)
print("Loading NLP model...")
nlp = spacy.load("en_core_web_sm")

results = []

print("\n" + "="*50)
print(" Starting date extraction and context retrieval from news articles...")
print("="*50)

N = 25

# Select N news at random
urls = df['News_URL'].dropna().sample(n=N, random_state=42).tolist()
df_w_texts = pd.DataFrame(columns = ['Event_Date', 'News_URL', 'Full_Text', 'Extracted_Dates', 'Context_Extracted','Obtained_Randomly'], index = range(N))

for i, url in enumerate(urls):
    print(f"\n Analyzing URL: {url[:60]}...")
    
    try:
        # 3. Download and parse the article using newspaper3k
        article = Article(url)
        article.download()
        article.parse()
        text = article.text
        
        if not text:
            print(" -> Error: Text extraction failed (possible paywall or empty content).")
            continue
            
        # 4. Process the text with spaCy's NLP to find sentences and entities
        doc = nlp(text)
        
        sentences_with_dates = []
        dates_found = set()
        
        # 5. Find sentences that contain DATE entities
        for sentence in doc.sents:
            has_date = False
            for entity in sentence.ents:
                if entity.label_ == "DATE": # If the AI detects it's a date
                    has_date = True
                    dates_found.add(entity.text)
            
            if has_date:
                # Save the full sentence, cleaned of newlines
                sentences_with_dates.append(sentence.text.replace('\n', ' ').strip())
        
        if sentences_with_dates:
            print(f" -> Success: Found {len(dates_found)} distinct dates.")
            df_w_texts.iloc[i] =  [df.loc[df['News_URL'] == url, 'Date'].values[0], url, text, ", ".join(dates_found), " | ".join(sentences_with_dates), True]
        else:
            print(" -> No results: No dates detected in the text.")
            
    except Exception as e:
        print(f" -> ERROR while scraping: The page blocked access or the link is broken.")



Loading NLP model...

 Starting date extraction and context retrieval from news articles...

 Analyzing URL: https://www.respublika.lt/lt/naujienos/nuomones_ir_komentara...
 -> No results: No dates detected in the text.

 Analyzing URL: https://www.heritage.org/europe/report/the-limits-glasnost...
 -> Success: Found 24 distinct dates.

 Analyzing URL: https://www.kfyrtv.com/2024/09/26/zelenskyy-is-visiting-whit...
 -> Success: Found 18 distinct dates.

 Analyzing URL: https://wiadomosci.dziennik.pl/polityka/artykuly/9446311,put...
 -> Success: Found 1 distinct dates.

 Analyzing URL: https://www.asahi.com/ajw/articles/15198284...
 -> Success: Found 4 distinct dates.

 Analyzing URL: https://war.obozrevatel.com/ukr/u-rosii-zayavili-scho-gotovi...
 -> Success: Found 3 distinct dates.

 Analyzing URL: https://www.rferl.org/a/putin-election-russia-war-repression...
 -> Success: Found 27 distinct dates.

 Analyzing URL: https://www.startribune.com/ukraine-aid-vote-is-a-domestic-a...
 -> ERR

In [107]:
# Select also news on relevant days and scrap them

totaL_top_counted_data = []
for country in daily_counts['Initiator_Country'].unique():
    counted_data = daily_counts[daily_counts['Initiator_Country'] == country].sort_values('News', ascending=False).head(10).reset_index(drop=True)
    totaL_top_counted_data.append(counted_data)

totaL_top_counted_data = pd.concat(totaL_top_counted_data, ignore_index=True).reset_index(drop=True)

df_w_texts_selected_dates = pd.DataFrame(columns = ['Event_Date', 'News_URL', 'Full_Text', 'Extracted_Dates', 'Context_Extracted', 'Obtained_Randomly'], index = range(len(totaL_top_counted_data)))

for index, row in totaL_top_counted_data.iterrows():
    date = row['Date']
    country = row['Initiator_Country']
    
    print(f"\n Scraping news for {country} on {date.strftime('%Y-%m-%d')}...")
    
    # Filter the original dataframe for this specific date and country
    relevant_articles = df[(df['Date'] == date) & (df['Initiator_Country'] == country)]
    
    # Get a random article from this filtered set
    art = relevant_articles.sample(n=1, random_state=42)
    url = art['News_URL'].values[0]
    print(f" -> Selected URL: {url[:60]}...")

    try:
        # Download and parse the article using newspaper3k
        article = Article(url)
        article.download()
        article.parse()
        text = article.text
        
        if not text:
            print(" -> Error: Text extraction failed (possible paywall or empty content).")
            continue
            
        # Process the text with spaCy's NLP to find sentences and entities
        doc = nlp(text)
        
        sentences_with_dates = []
        dates_found = set()
        
        # Find sentences that contain DATE entities
        for sentence in doc.sents:
            has_date = False
            for entity in sentence.ents:
                if entity.label_ == "DATE": # If the AI detects it's a date
                    has_date = True
                    dates_found.add(entity.text)
            
            if has_date:
                # Save the full sentence, cleaned of newlines
                sentences_with_dates.append(sentence.text.replace('\n', ' ').strip())
        
        if sentences_with_dates:
            print(f" -> Success: Found {len(dates_found)} distinct dates.")
            df_w_texts_selected_dates.iloc[index] =  [date, url, text, ", ".join(dates_found), " | ".join(sentences_with_dates), False]
        else:

            print(" -> Error: No relevant dates found.")

    except Exception as e:
        print(f" -> Error: Failed to process article at {url}. Exception: {e}")



 Scraping news for CHN on 2024-03-14...
 -> Selected URL: https://dnes.dir.bg/comments/prouchvane-v-sasht-baydan-drapn...
 -> Error: No relevant dates found.

 Scraping news for CHN on 2024-04-24...
 -> Selected URL: https://www.lemonde.fr/pixels/article/2024/04/24/tiktok-les-...
 -> Error: Failed to process article at https://www.lemonde.fr/pixels/article/2024/04/24/tiktok-les-questions-que-pose-sa-possible-interdiction-aux-etats-unis_6229649_4408996.html. Exception: Article `download()` failed with 406 Client Error: Not Acceptable for url: https://www.lemonde.fr/pixels/article/2024/04/24/tiktok-les-questions-que-pose-sa-possible-interdiction-aux-etats-unis_6229649_4408996.html on URL https://www.lemonde.fr/pixels/article/2024/04/24/tiktok-les-questions-que-pose-sa-possible-interdiction-aux-etats-unis_6229649_4408996.html

 Scraping news for CHN on 2024-03-13...
 -> Selected URL: https://www.thesunchronicle.com/ap/business/house-passes-a-b...
 -> Error: Failed to process article at h

In [108]:
joined_df = pd.concat([df_w_texts.dropna(), df_w_texts_selected_dates.dropna()], ignore_index=True)
joined_df.to_excel('articles_with_texts.xlsx', index=False)

C:\Users\danie\AppData\Local\Temp\ipykernel_19184\2152446540.py:2: UserWarning: Cell contents too long (100000), truncated to 32767 characters
  joined_df.to_excel('articles_with_texts.xlsx', index=False)
C:\Users\danie\AppData\Local\Temp\ipykernel_19184\2152446540.py:2: UserWarning: Cell contents too long (43053), truncated to 32767 characters
  joined_df.to_excel('articles_with_texts.xlsx', index=False)


In [109]:
joined_df[['Event_Date', 'News_URL', 'Full_Text']].to_excel('articles_with_texts_selected_columns.xlsx', index=False)

C:\Users\danie\AppData\Local\Temp\ipykernel_19184\2026683881.py:1: UserWarning: Cell contents too long (100000), truncated to 32767 characters
  joined_df[['Event_Date', 'News_URL', 'Full_Text']].to_excel('articles_with_texts_selected_columns.xlsx', index=False)
C:\Users\danie\AppData\Local\Temp\ipykernel_19184\2026683881.py:1: UserWarning: Cell contents too long (43053), truncated to 32767 characters
  joined_df[['Event_Date', 'News_URL', 'Full_Text']].to_excel('articles_with_texts_selected_columns.xlsx', index=False)


#### Using a LLM for question-answering

In [21]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

# 1. Load the DeBERTa model (The best model for tricky reading comprehension)
print("Loading the local AI model...")
model_name = "deepset/deberta-v3-base-squad2" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# 2. The core extraction engine
def ask_ai(question, context, threshold=0.0):
    try:
        inputs = tokenizer(question, context, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            outputs = model(**inputs)
            
        start_idx = outputs.start_logits.argmax()
        end_idx = outputs.end_logits.argmax()
        score = outputs.start_logits[0, start_idx].item() + outputs.end_logits[0, end_idx].item()
        
        if end_idx >= start_idx and score > threshold:
            answer_tokens = inputs.input_ids[0, start_idx : end_idx + 1]
            extracted = tokenizer.decode(answer_tokens, skip_special_tokens=True)
            if not extracted.strip() or extracted in ["<s>", "[CLS]"]:
                return "UNCLEAR", score
            return extracted, score
        else:
            return "UNCLEAR", score
    except Exception:
        return "ERROR", -999.0

# 3. The Prompt Ensembling Function
def get_best_date(actor_name, context):
    """
    Asks multiple differently phrased questions for the same actor.
    Returns the answer with the highest confidence score.
    """
    questions = [
        f"In what month and year did the {actor_name} attack or interference occur?",
        f"When did {actor_name} actually hack or breach the system?",
        f"What is the date of the {actor_name} cyberattack?",
        f"When did the {actor_name} operation take place?"
    ]
    
    best_answer = "UNCLEAR"
    highest_score = -999.0
    
    for q in questions:
        ans, score = ask_ai(q, context, threshold=0.0)
        
        # If we found an answer and its score is the highest we've seen so far, save it
        if ans != "UNCLEAR" and score > highest_score:
            best_answer = ans
            highest_score = score
            
    return best_answer, highest_score

# 4. Load your scraped contexts
file_name = "Contexto_Fechas_Extraidas.xlsx"
try:
    df = pd.read_excel(file_name)
except FileNotFoundError:
    print(f"Error: Make sure '{file_name}' is in the same folder!")
    exit()

print(f"\nLoaded {len(df)} rows. Starting Multi-Prompt extraction...\n")

dates_china, dates_russia, dates_iran = [], [], []

# 5. Loop through the data
for index, row in df.iterrows():
    context = str(row['Contexto_Extraido'])
    
    if context == "nan" or context.strip() == "":
        dates_china.append("NO CONTEXT")
        dates_russia.append("NO CONTEXT")
        dates_iran.append("NO CONTEXT")
        continue
        
    print(f"\n--- Row {index + 1} ---")
    print(f"Context: {context[:80]}...")
    
    # Run the prompt ensemble for each actor
    ans_chn, score_chn = get_best_date("Chinese", context)
    print(f" -> China Best:  {ans_chn} (Score: {score_chn:.1f})")
    dates_china.append(ans_chn)
    
    ans_rus, score_rus = get_best_date("Russian", context)
    print(f" -> Russia Best: {ans_rus} (Score: {score_rus:.1f})")
    dates_russia.append(ans_rus)
    
    ans_irn, score_irn = get_best_date("Iranian", context)
    print(f" -> Iran Best:   {ans_irn} (Score: {score_irn:.1f})")
    dates_iran.append(ans_irn)

# 6. Save the final results
df['Date_China'] = dates_china
df['Date_Russia'] = dates_russia
df['Date_Iran'] = dates_iran

output_file = 'Ensemble_Actor_Timelines.xlsx'
df.to_excel(output_file, index=False)

print(f"\nExtraction complete! Check '{output_file}'.")

c:\Users\danie\anaconda3\envs\interference\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading the local AI model...


Loading weights: 100%|██████████| 200/200 [00:00<00:00, 1992.15it/s]
DebertaV2ForQuestionAnswering LOAD REPORT from: deepset/deberta-v3-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Loaded 16 rows. Starting Multi-Prompt extraction...


--- Row 1 ---
Context: In September, the FBI announced that it had disrupted a vast Chinese hacking ope...
 -> China Best:  September (Score: 9.8)
 -> Russia Best: UNCLEAR (Score: -999.0)
 -> Iran Best:   UNCLEAR (Score: -999.0)

--- Row 2 ---
Context: In September, the FBI announced that it had disrupted a vast Chinese hacking ope...
 -> China Best:  September (Score: 9.8)
 -> Russia Best: UNCLEAR (Score: -999.0)
 -> Iran Best:   UNCLEAR (Score: -999.0)

--- Row 3 ---
Context: The two sides will hold “intensive exchanges” on more balanced economic growth, ...
 -> China Best:  UNCLEAR (Score: -999.0)
 -> Russia Best: UNCLEAR (Score: -999.0)
 -> Iran Best:   UNCLEAR (Score: -999.0)

--- Row 4 ---
Context: A Chinese artisan makes a rice sculpture for visitors at the Philadelphia Museum...
 -> China Best:  Feb. 12 (Score: 0.3)
 -> Russia Best: UNCLEAR (Score: -999.0)
 -> Iran Best:   UNCLEAR (Score: -999.0)

--- Row 5 ---
Context: Tid

KeyboardInterrupt: 

'As Donald Trump flew to what would be his ruinous performance at Tuesday’s presidential debate, he hosted a special guest on his private plane — Laura Loomer. | That’s a smackdown from Loomer’s former friend and close ally, mega-MAGA Georgia Rep. Marjorie Taylor Greene, herself a major proponent of that vile slur about Haitian immigrants in Ohio supposedly kidnapping and eating house pets — the one Trump keeps incoherently repeating this week. | They eat HUMANS,” she posted on X Thursday. | Thursday, he “went off on ‘toxic’ Laura Loomer and advised (Donald) Trump to ‘make sure this doesn’t become a bigger story,” according to Punchbowl News senior congressional reporter Andrew Desiderio. | Because Loomer is at Trump’s side now, as she was at an observance of the Sept. 11, 2001, terrorist attacks — events she calls “an inside job.”  Be worried. | Watching Trump’s earnest, angry blather Tuesday night about people eating cats and “transgender operations on illegal aliens that are in pris